```
data
    train
        calss1
            img1.jpg
            img2.jpg
        class2
            img1.jpg
            img2.jpg
```


In [2]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((32,32)), #이미지 사이즈 통일
    transforms.ToTensor(), #텐서로 변환
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)) #정규화
])

train_dataset = datasets.ImageFolder(root='./data/train',transform=transform)
test_dataset = datasets.ImageFolder(root='./data/validation',transform=transform)
train_loader = DataLoader(train_dataset, batch_size=5, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=5, shuffle=False)

In [3]:
img, label = next(iter(train_loader))
label

tensor([0, 0, 1, 1, 1])

In [8]:
img.size()

torch.Size([5, 3, 32, 32])

In [10]:
print(train_dataset.classes)
print(train_dataset.class_to_idx)

['horses', 'humans']
{'horses': 0, 'humans': 1}


In [11]:
import torch.nn as nn

class BasicBlock(nn.Module):
    
    def __init__(self, int_channels, out_channels, hidden_dim):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(int_channels, hidden_dim, kernel_size=3,padding=1)
        self.conv2 = nn.Conv2d(hidden_dim, out_channels,kernel_size=3,padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2)
    
    def forward(self, x):
        x = self.relu( self.conv1(x) )
        x = self.relu( self.conv2(x) )
        out = self.pool(x)
        return out

In [14]:
class CNN(nn.Module):

    def __init__(self, num_class):
        super(CNN, self).__init__( )
        self.block1 = BasicBlock(3,64,64)
        self.block2 = BasicBlock(64,128,128) #in: 위의 out과 맞추기
        
        # 분류기
        self.fc1 = nn.Linear(128*8*8, 2048) #in 직접 계산해줘야 함.
        self.fc2 = nn.Linear(2048, 256)
        self.fc3 = nn.Linear(256, num_class)
        self.relu = nn.ReLU() #기울기 소실 방지!
    
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        # (-1, 128*8*8)
        x = torch.flatten(x, start_dim=1)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        out = self.fc3(x)
        return out

In [17]:
X = torch.randn(5,3,32,32)
model = CNN(2)
model(X)

tensor([[-0.0481, -0.0125],
        [-0.0501, -0.0128],
        [-0.0489, -0.0125],
        [-0.0477, -0.0107],
        [-0.0472, -0.0138]], grad_fn=<AddmmBackward0>)

In [20]:
from torch.optim import Adam
from tqdm import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = CNN(10)
model.to(device)

#---- 2. 학습루프 ----
lr = 1e-3
optim = Adam(model.parameters(), lr=lr)
epochs = 50

for epoch in tqdm(range(epochs)):
    for data, label in train_loader:
        optim.zero_grad()
        preds = model(data.to(device))
        loss = nn.CrossEntropyLoss()(preds,label.to(device))
        loss.backward()
        optim.step()
    if (epoch+1) % 10 == 0:
        print(f'epoch: {epoch+1} | loss: {loss.item():.4f}')

 20%|██        | 10/50 [01:07<04:42,  7.07s/it]

epoch: 10 | loss: 0.1236


 40%|████      | 20/50 [02:36<04:21,  8.73s/it]

epoch: 20 | loss: 0.0000


 60%|██████    | 30/50 [03:48<02:31,  7.56s/it]

epoch: 30 | loss: 0.0000


 80%|████████  | 40/50 [05:34<01:56, 11.64s/it]

epoch: 40 | loss: 0.0000


100%|██████████| 50/50 [07:11<00:00,  8.64s/it]

epoch: 50 | loss: 0.0000


In [21]:
# 평가

# 예측
num_corr = 0
model.eval()

with torch.no_grad():
    for data, label in test_loader:
        output = model(data.to(device))
        preds = output.data.max(1)[1]
        corr = preds.eq(label.to(device).data).sum().item()
        num_corr += corr
    print(f'Accuracy: {num_corr / len(test_loader)}')

Accuracy: 1.7894736842105263
